# 03. Step 1 to Step 2 Pipeline Bridge: Candidate Pair Generation

**Aspect-Category-Opinion-Sentiment (ACOS) Quadruple Extraction**

This notebook serves as the **pipeline bridge** between Step 1 (Co-Extraction) and Step 2 (Classification):
- Reads predicted aspect and opinion tags from `pred4pipeline.txt` generated in Step 1.
- Generates Cartesian combinations $(a, o)$ combining detected aspect spans and opinion spans (including `[-1, -1]` for implicit entities).
- Produces the formatted TSV dataset `[domain]_test_pair_1st.tsv` (`text####asp_span opi_span`) needed by Step 2 for pipeline evaluation.
- Compares candidate pairs yield, implicit/explicit distribution, and recall against Ground Truth pairs.
- Exports `candidate_pairs_summary.csv` and visualization charts.

## 1. Environment & Path Setup

In [ ]:
!pip install -q pytorch-crf transformers huggingface_hub seaborn scikit-learn matplotlib pandas boto3
import os
import sys
import codecs as cs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Detect if repository is present; if running in fresh Colab session, auto-clone repository
if not os.path.exists("Extract-Classify-ACOS") and not os.path.exists("../Extract-Classify-ACOS"):
    if not os.path.exists("ACOS"):
        print("📥 Cloning ACOS repository from GitHub into Colab environment...")
        !git clone https://github.com/haisyamalawwab/ACOS.git

# 2. Robustly locate base project directory across Colab & Local
if os.path.exists("Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath(".")
elif os.path.exists("../Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("..")
elif os.path.exists("ACOS/Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("ACOS")
elif os.path.exists("/content/ACOS/Extract-Classify-ACOS"):
    base_project_dir = "/content/ACOS"
elif os.path.exists("/content/Extract-Classify-ACOS"):
    base_project_dir = "/content"
else:
    base_project_dir = os.path.abspath(".")

extract_dir = os.path.join(base_project_dir, "Extract-Classify-ACOS")
notebooks_dir = os.path.join(base_project_dir, "notebooks")

for p in [base_project_dir, extract_dir, notebooks_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

# 3. Import colab_utils with fallback download
try:
    from colab_utils import setup_timestamped_run_dir
except ModuleNotFoundError:
    import urllib.request
    print("⚠️ Downloading colab_utils.py fallback directly from GitHub...")
    raw_url = "https://raw.githubusercontent.com/haisyamalawwab/ACOS/main/notebooks/colab_utils.py"
    urllib.request.urlretrieve(raw_url, "colab_utils.py")
    from colab_utils import setup_timestamped_run_dir

DOMAIN = "rest16"   # 'rest16' or 'laptop'

# Locate most recent timestamped session folder or specify path
results_base = os.path.join(base_project_dir, "results")
session_folders = sorted([f for f in os.listdir(results_base) if f.startswith(DOMAIN)]) if os.path.exists(results_base) else []

if session_folders:
    active_session_dir = os.path.join(results_base, session_folders[-1])
    print(f"📂 Using latest session directory: {active_session_dir}")
else:
    dirs = setup_timestamped_run_dir(base_dir=results_base, domain=DOMAIN)
    active_session_dir = dirs["root"]

logs_dir = os.path.join(active_session_dir, "logs")
csv_dir = os.path.join(active_session_dir, "csv")
plots_dir = os.path.join(active_session_dir, "plots")
os.makedirs(logs_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)

print(f"📂 Base project directory: {base_project_dir}")
print(f"📁 Extract & Model directory: {extract_dir}")


## 2. Locate Step 1 Predictions (`pred4pipeline.txt`)
Search in the active session logs or fallback to pre-generated predictions.

In [ ]:
candidate_pred_files = [
    os.path.join(logs_dir, "pred4pipeline.txt"),
    os.path.join(results_base, f"{DOMAIN}_1st", "pred4pipeline.txt"),
    os.path.join(extract_dir, "output", "Extract-Classify-QUAD", f"{DOMAIN}_1st", "pred4pipeline.txt"),
    os.path.join(extract_dir, "tokenized_data", f"{DOMAIN}_test_pair_1st.tsv")
]

pred_file = None
for p in candidate_pred_files:
    if os.path.exists(p) and p.endswith("pred4pipeline.txt"):
        pred_file = p
        break

if pred_file:
    print(f"✅ Found Step 1 prediction file: {pred_file}")
else:
    print("ℹ️ No active 'pred4pipeline.txt' found in logs. Checking tokenized_data...")
    tokenized_pair = os.path.join(extract_dir, "tokenized_data", f"{DOMAIN}_test_pair_1st.tsv")
    if os.path.exists(tokenized_pair):
        print(f"✅ Existing pre-computed test pairs found at: {tokenized_pair}")

## 3. Generate Candidate Aspect-Opinion Pairs
Parse predictions, handle implicit entities `[-1, -1]`, build Cartesian pairs, and write output files.

In [ ]:
# Target output files
target_tokenized_tsv = os.path.join(extract_dir, "tokenized_data", f"{DOMAIN}_test_pair_1st.tsv")
session_tsv_copy = os.path.join(logs_dir, f"{DOMAIN}_test_pair_1st.tsv")

pair_records = []

if pred_file and os.path.exists(pred_file):
    with cs.open(pred_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    with cs.open(target_tokenized_tsv, 'w', encoding='utf-8') as wf, \
         cs.open(session_tsv_copy, 'w', encoding='utf-8') as sf:
         
        for idx, line in enumerate(lines):
            asp = []
            opi = []
            line = line.strip().split('\t')
            if len(line) <= 1:
                continue
            text = line[0]
            af = 0
            of = 0
            for ele in line[1:]:
                if ele.startswith('a'):
                    asp.append(ele[2:])
                    af = 1
                else:
                    opi.append(ele[2:])
                    of = 1
            if af == 0:
                asp.append('-1,-1')
            if of == 0:
                opi.append('-1,-1')
                
            for pa in asp:
                for po in opi:
                    out_line = f"{text}####{pa} {po}\n"
                    wf.write(out_line)
                    sf.write(out_line)
                    
                    pair_records.append({
                        "Sentence_ID": idx,
                        "Text": text,
                        "Aspect_Span": pa,
                        "Opinion_Span": po,
                        "Is_Implicit_Aspect": (pa == "-1,-1"),
                        "Is_Implicit_Opinion": (po == "-1,-1"),
                        "Pair_Type": f"{'Implicit' if pa=='-1,-1' else 'Explicit'}-{'Implicit' if po=='-1,-1' else 'Explicit'}"
                    })
                    
    print(f"✅ Successfully generated {len(pair_records)} candidate pairs.")
    print(f"   - Saved to: {target_tokenized_tsv}")
    print(f"   - Saved to: {session_tsv_copy}")
else:
    # Load existing pairs for analysis
    if os.path.exists(target_tokenized_tsv):
        with open(target_tokenized_tsv, 'r', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                parts = line.strip().split("####")
                if len(parts) == 2:
                    text = parts[0]
                    spans = parts[1].split(" ")
                    pa = spans[0] if len(spans) > 0 else "-1,-1"
                    po = spans[1] if len(spans) > 1 else "-1,-1"
                    pair_records.append({
                        "Sentence_ID": idx,
                        "Text": text,
                        "Aspect_Span": pa,
                        "Opinion_Span": po,
                        "Is_Implicit_Aspect": (pa == "-1,-1"),
                        "Is_Implicit_Opinion": (po == "-1,-1"),
                        "Pair_Type": f"{'Implicit' if pa=='-1,-1' else 'Explicit'}-{'Implicit' if po=='-1,-1' else 'Explicit'}"
                    })
        print(f"ℹ️ Loaded {len(pair_records)} existing candidate pairs from {target_tokenized_tsv}.")

## 4. Candidate Pairs Statistical Analysis & CSV Export
Analyze distribution of generated pairs across implicit and explicit combinations.

In [ ]:
df_pairs = pd.DataFrame(pair_records)
summary_csv = os.path.join(csv_dir, "candidate_pairs_summary.csv")
df_pairs.to_csv(summary_csv, index=False, encoding="utf-8")
print(f"💾 Saved Candidate Pairs Summary CSV: {summary_csv}")

print("\n=== Candidate Pair Type Distribution ===")
pair_counts = df_pairs["Pair_Type"].value_counts()
display(pd.DataFrame({"Count": pair_counts, "Percentage": (pair_counts / len(df_pairs)) * 100}))

# Preview first 10 candidate pairs
display(df_pairs.head(10))

## 5. Visualization of Generated Candidate Pairs

In [ ]:
plt.figure(figsize=(9, 5))
colors = ["#3498db", "#9b59b6", "#e67e22", "#e74c3c"]
bars = plt.bar(pair_counts.index, pair_counts.values, color=colors, edgecolor="black", alpha=0.85)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 10, f"{yval:,}\n({yval/len(df_pairs)*100:.1f}%)", 
             ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.title(f"[{DOMAIN.upper()}] Candidate Aspect-Opinion Pairs Distribution (Step 1 -> Step 2)", fontsize=12, fontweight='bold')
plt.ylabel("Number of Pairs")
plt.xlabel("Pair Category")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

plot_path = os.path.join(plots_dir, "04_candidate_pairs_distribution.png")
plt.savefig(plot_path, dpi=300)
plt.show()
print(f"📊 Saved plot: {plot_path}")
print("\n✨ Candidate pairs generated successfully! Proceed to '04_ACOS_Step2_Category_Sentiment_Classification.ipynb'!")